# Final Table Creation Combining MP Look Up + 28th Term Speech Level Tables

Bu notebook, konuşmacı temizleme ve milletvekili eşleştirme yaklaşımını tek bir akışta toplar. Final tablo şu alanları içerir:

`session_id`, `donem`, `yasama_yili`, `birlesim_no`, `date`, kaynak verideki bütün URL sütunları, `konusmaci_raw`, `konusmaci_ad_soyad`, `konusma_text`, `konusma_sirasi`, `konusma_uzunlugu`, `kelime_sayisi`, `konusmaci_parti`, `konusmaci_sehir`, `konusmaci_tip`.

`başkan`, `bakan`, `cumhurbaşkanı` gibi seçim çevresi bilgisi kullanılmaması gereken görevlerde `konusmaci_sehir` özellikle `NA` bırakılır.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import re
import unicodedata

import pandas as pd
from IPython.display import display

# Colab/yerel ortamınıza göre yalnızca bu yolları değiştirin.
INPUT_CSV = Path('/content/drive/MyDrive/tbmm_raw_tutanaklar/tutanaklar_28_konusmaci_bazli.csv')
LOOKUP_PATH = Path('/content/drive/MyDrive/tbmm_raw_tutanaklar/milletvekili_party_city_lookup.csv')
OUTPUT_CSV = Path('/content/drive/MyDrive/tbmm_raw_tutanaklar/tbmm_konusma_final_table.csv')

# Lookup dosyası isteğe bağlıdır. Dosya bulunamazsa ham etikette yazan parti/şehir
# bilgileri yine kullanılır; yalnızca lookup ile doldurulabilecek alanlar NA kalır.
print('Input :', INPUT_CSV)
print('Lookup:', LOOKUP_PATH)
print('Output:', OUTPUT_CSV)

Input : /content/drive/MyDrive/tbmm_raw_tutanaklar/tutanaklar_28_konusmaci_bazli.csv
Lookup: /content/drive/MyDrive/tbmm_raw_tutanaklar/milletvekili_party_city_lookup.csv
Output: /content/drive/MyDrive/tbmm_raw_tutanaklar/tbmm_konusma_final_table.csv


In [ ]:
df = pd.read_csv(INPUT_CSV, low_memory=False)

required_input_columns = [
    'session_id', 'donem', 'yasama_yili', 'birlesim_no', 'date',
    'konusmaci_raw', 'konusma_text'
]
missing_input_columns = [c for c in required_input_columns if c not in df.columns]
if missing_input_columns:
    raise ValueError(f'Girdi dosyasında eksik zorunlu sütunlar var: {missing_input_columns}')

# Eski kaynaklarda bu üç alan yoksa güvenli biçimde üret.
if 'konusma_sirasi' not in df.columns:
    df['konusma_sirasi'] = df.groupby('session_id', dropna=False).cumcount() + 1

speech_text = df['konusma_text'].fillna('').astype(str)
if 'konusma_uzunlugu' not in df.columns:
    df['konusma_uzunlugu'] = speech_text.str.len()
if 'kelime_sayisi' not in df.columns:
    df['kelime_sayisi'] = speech_text.str.findall(r'\S+').str.len()

url_columns = [c for c in df.columns if 'url' in str(c).casefold()]

print(f'Satır sayısı: {len(df):,}')
print('URL sütunları:', url_columns)
display(df.head(3))

Satır sayısı: 199,158
URL sütunları: ['list_page_url', 'ham_page_url', 'ozet_page_url', 'ham_html_url']


,session_id,donem,yasama_yili,birlesim_no,date,list_page_url,ham_page_url,ozet_page_url,ham_html_url,konusmaci_raw,konusma_text,konusma_sirasi,konusma_uzunlugu,kelime_sayisi
0,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,Türkiye Büyük Millet Meclisinin 28’inci Yasama...,1,138,17
1,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,"Sayın milletvekilleri, Sayın Cumhurbaşkanı ken...",2,314,35
2,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,"Sayın milletvekilleri, şimdi gündeme geçiyoruz...",3,196,24


In [ ]:
# -----------------------------
# Metin ve anahtar normalizasyonu
# -----------------------------
MOJIBAKE_REPLACEMENTS = {
    'Ã‡': 'Ç', 'Äž': 'Ğ', 'Ä°': 'İ', 'Ã–': 'Ö', 'Åž': 'Ş', 'Ãœ': 'Ü',
    'Ã§': 'ç', 'ÄŸ': 'ğ', 'Ä±': 'ı', 'Ã¶': 'ö', 'ÅŸ': 'ş', 'Ã¼': 'ü',
    'Ã‚': 'Â', 'Ã¢': 'â', 'ÃŽ': 'Î', 'Ã®': 'î', 'Ã›': 'Û', 'Ã»': 'û',
    'â€™': '’', 'â€œ': '“', 'â€': '”', 'â€“': '–', 'â€”': '—'
}
TR_UPPER_TRANS = str.maketrans({'i': 'İ', 'ı': 'I'})

def repair_mojibake(value):
    """Yaygın UTF-8/cp1252 bozulmasını yalnızca ayrıştırma için düzeltir."""
    if pd.isna(value):
        return ''
    text = str(value)
    for broken, repaired in MOJIBAKE_REPLACEMENTS.items():
        text = text.replace(broken, repaired)
    return text

def norm_space(value):
    text = repair_mojibake(value).replace('\xa0', ' ')
    return re.sub(r'\s+', ' ', text).strip()

def tr_upper(value):
    return norm_space(value).translate(TR_UPPER_TRANS).upper()

def clean_key(value):
    text = unicodedata.normalize('NFKC', tr_upper(value))
    text = re.sub(r'[^\wÇĞİÖŞÜÂÎÛ ]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


# -----------------------------
# Şehir normalizasyonu
# -----------------------------
PROVINCES = [
    'Adana', 'Adıyaman', 'Afyonkarahisar', 'Ağrı', 'Amasya', 'Ankara',
    'Antalya', 'Artvin', 'Aydın', 'Balıkesir', 'Bilecik', 'Bingöl',
    'Bitlis', 'Bolu', 'Burdur', 'Bursa', 'Çanakkale', 'Çankırı', 'Çorum',
    'Denizli', 'Diyarbakır', 'Edirne', 'Elazığ', 'Erzincan', 'Erzurum',
    'Eskişehir', 'Gaziantep', 'Giresun', 'Gümüşhane', 'Hakkâri', 'Hatay',
    'Isparta', 'Mersin', 'İstanbul', 'İzmir', 'Kars', 'Kastamonu',
    'Kayseri', 'Kırklareli', 'Kırşehir', 'Kocaeli', 'Konya', 'Kütahya',
    'Malatya', 'Manisa', 'Kahramanmaraş', 'Mardin', 'Muğla', 'Muş',
    'Nevşehir', 'Niğde', 'Ordu', 'Rize', 'Sakarya', 'Samsun', 'Siirt',
    'Sinop', 'Sivas', 'Tekirdağ', 'Tokat', 'Trabzon', 'Tunceli',
    'Şanlıurfa', 'Uşak', 'Van', 'Yozgat', 'Zonguldak', 'Aksaray',
    'Bayburt', 'Karaman', 'Kırıkkale', 'Batman', 'Şırnak', 'Bartın',
    'Ardahan', 'Iğdır', 'Yalova', 'Karabük', 'Kilis', 'Osmaniye', 'Düzce'
]
CITY_MAP = {tr_upper(city): city for city in PROVINCES}
CITY_ALIASES = {
    'AFYON': 'Afyonkarahisar',
    'ELÂZIĞ': 'Elazığ',
    'ELAZIĞ': 'Elazığ',
    'HAKKARİ': 'Hakkâri',
    'İÇEL': 'Mersin',
    'ISTANBUL': 'İstanbul',
    'IZMIR': 'İzmir',
    'SANLIURFA': 'Şanlıurfa',
}
DEVAM_WORDS = {'DEVAM', 'DEVAMLA', 'DEVAMEN', 'DEVAM EDEREK'}

def canonical_city(value):
    text = norm_space(value).strip(' .')
    upper = tr_upper(text)
    if not text or upper in DEVAM_WORDS:
        return pd.NA
    return CITY_ALIASES.get(upper, CITY_MAP.get(upper, pd.NA))

def extract_city_from_raw(raw):
    for parenthetical in reversed(re.findall(r'\(([^()]*)\)', norm_space(raw))):
        city = canonical_city(parenthetical)
        if pd.notna(city):
            return city
    return pd.NA


# -----------------------------
# Parti normalizasyonu
# -----------------------------
PARTY_PATTERNS = [
    (r'\bADALET VE KALKINMA PART[İI]S[İI]\b|\bAK\s*PART[İI]\b', 'AK PARTİ'),
    (r'\bCUMHUR[İI]YET HALK PART[İI]S[İI]\b|\bCHP\b', 'CHP'),
    (r'\bM[İI]LL[İI]YETÇ[İI] HAREKET PART[İI]S[İI]\b|\bMHP\b', 'MHP'),
    (r'\b[İI]Y[İI]\s+PART[İI]\b', 'İYİ PARTİ'),
    (r'\bDEMOKRAT[İI]K SOL PART[İI]\b|\bDSP\b', 'DSP'),
    (r'\bDEMOKRAS[İI] VE ATILIM PART[İI]S[İI]\b|\bDEVA\b', 'DEVA PARTİSİ'),
    (r'\bDEMOKRAT PART[İI]\b', 'DEMOKRAT PARTİ'),
    (r'\bHALKLARIN EŞ[İI]TL[İI]K VE DEMOKRAS[İI] PART[İI]S[İI]\b|\bHEDEP\b', 'HEDEP / DEM PARTİ'),
    (r'\bDEM\s+PART[İI]?\b', 'DEM PARTİ'),
    (r'\bYEŞ[İI]L SOL PART[İI](?:S[İI])?\b', 'YEŞİL SOL PARTİ'),
    (r'\bSAADET PART[İI]S[İI]\b', 'SAADET PARTİSİ'),
    (r'\bYEN[İI]\s+YOL(?:\s+PART[İI]S[İI])?\b', 'YENİ YOL PARTİSİ'),
    (r'\bTÜRK[İI]YE [İI]ŞÇ[İI] PART[İI]S[İI]\b|\bT[İI]P\b', 'TİP'),
    (r'\bHÜR DAVA PART[İI]S[İI]\b|\bHÜDA PAR\b', 'HÜDA PAR'),
    (r'\bYEN[İI]DEN REFAH PART[İI]S[İI]\b', 'YENİDEN REFAH PARTİSİ'),
    (r'\bEMEK PART[İI]S[İI]\b|\bEMEP\b', 'EMEP'),
    (r'\bGELECEK PART[İI]S[İI]\b', 'GELECEK PARTİSİ'),
]

def normalize_party(value):
    upper = tr_upper(value)
    for pattern, party in PARTY_PATTERNS:
        if re.search(pattern, upper):
            return party
    return pd.NA


# -----------------------------
# Konuşmacı türü ve ad-soyad ayrıştırma
# -----------------------------
BAD_EXACT_LABELS = {
    'A.', 'B', 'B.', 'II.', 'III.', 'VII', 'ABD', 'AKP', 'BAĞ', 'DGD',
    'DİSK', 'GİNE', 'HIT', 'IŞİD', 'PAN', 'PKK', 'TRAB', 'TÜRK'
}
BAD_PHRASES = (
    'GEÇİCİ MADDE', 'TÜRKİYE CUMHURİYETİ HÜKÜMETİ İLE',
    'ÜNİVERSİTESİ', 'EĞİTİM SEN', 'DİSK GENEL'
)

def looks_like_bad_label(raw):
    upper = tr_upper(raw).strip()
    if not upper or upper in BAD_EXACT_LABELS:
        return True
    if any(phrase in upper for phrase in BAD_PHRASES):
        return True
    compact = re.sub(r'\W+', '', upper)
    return len(compact) <= 4 and '(' not in upper and not upper.startswith(('BAŞKAN', 'TBMM'))

GROUP_PREFIX_PATTERN = (
    r'^(?:AK\s*PART[İI]|CHP|DEM\s+PART[İI]?|HEDEP|YEŞ[İI]L\s+SOL\s+PART[İI](?:S[İI])?|'
    r'MHP|[İI]Y[İI]\s+PART[İI]|SAADET\s+PART[İI]S[İI]|YEN[İI]\s+YOL(?:\s+PART[İI]S[İI])?)\s+'
    r'(?:MECL[İI]S\s+)?(?:GRUBU|GRUBA|GBURU|GBUBU|GURUBU|GRUP|GRUBUNA)\s*(?:ADINA)?\s+'
)

def is_group_speech(raw):
    return bool(re.search(r'(GRUBU|GRUBA|GBURU|GBUBU|GURUBU|GRUBUNA|GRUP)\s*(ADINA)?', tr_upper(raw)))

def classify_speaker_type(raw):
    upper = tr_upper(raw)
    if looks_like_bad_label(raw):
        return 'parse_error'
    if upper.startswith('TBMM BAŞKANI'):
        return 'tbmm başkanı'
    if upper.startswith(('BAŞKAN VEKİLİ', 'BAŞKANVEKİLİ')):
        return 'başkan vekili'
    if upper.startswith('BAŞKAN '):
        return 'başkan'
    if upper.startswith('CUMHURBAŞKANI YARDIMCISI'):
        return 'cumhurbaşkanı yardımcısı'
    if upper.startswith('CUMHURBAŞKANI '):
        return 'cumhurbaşkanı'
    if 'DEVLET BAŞKANI' in upper:
        return 'devlet başkanı'
    if 'BAŞDENETÇİSİ' in upper:
        return 'kamu başdenetçisi'
    if re.search(r'\bBAKANI\b', upper):
        return 'bakan'
    if re.search(r'KOM[İI]SYON\w*\s+BAŞKAN\s*VEK[İI]L[İI]', upper):
        return 'komisyon başkan vekili'
    if re.search(r'KOM[İI]SYON\w*\s+BAŞKAN(?:I)?\b', upper):
        return 'komisyon başkanı'
    if re.search(r'KOM[İI]SYON\w*\s+SÖZCÜSÜ|KOM[İI]SYON\w*\s+SÖZÜSÜ', upper):
        return 'komisyon sözcüsü'
    if re.search(r'GENEL BAŞKANI\s+ADINA', upper):
        return 'parti genel başkanı adına'
    if re.search(r'EŞ\s+GENEL BAŞKANI|GENEL BAŞKANI', upper):
        return 'parti genel başkanı'
    if re.search(r'GRUP BAŞKAN VEK[İI]L[İI]', upper):
        return 'grup başkan vekili'
    if re.search(r'MECL[İI]S GRUBU BAŞKANI|GRUBU BAŞKANI', upper):
        return 'meclis grubu başkanı'
    if is_group_speech(raw):
        return 'milletvekili (grup adına)'
    return 'milletvekili'

def strip_prefix_using_upper(original, pattern):
    text = norm_space(original)
    match = re.match(pattern, tr_upper(text))
    return norm_space(text[match.end():]) if match else text

PREFIX_PATTERNS = [
    r'^SAYIN\s+',
    r'^TBMM\s+BAŞKANI\s+',
    r'^BAŞKAN\s+VEK[İI]L[İI]\s+',
    r'^BAŞKANVEK[İI]L[İI]\s+',
    r'^BAŞKAN\s+',
    r'^CUMHURBAŞKANI\s+YARDIMCISI\s+',
    r'^CUMHURBAŞKANI\s+',
    r'^F[İI]L[İI]ST[İI]N\s+DEVLET\s+BAŞKANI\s+',
    r'^KAMU\s+BAŞDENETÇ[İI]S[İI]\s+',
    r'^.*?\bKOM[İI]SYON\w*\s+(?:BAŞKAN\s*VEK[İI]L[İI]|BAŞKANVEK[İI]L[İI]|BAŞKAN(?:I)?|SÖZCÜSÜ|SÖZÜSÜ)\s+',
    r'^.*?\bBAKANI\s+',
    r'^.*?\b(?:EŞ\s+)?GENEL\s+BAŞKANI(?:\s+VE\s+MECL[İI]S\s+GRUBU\s+BAŞKANI)?(?:\s+ADINA)?\s+',
    r'^.*?\b(?:MECL[İI]S\s+)?GRUBU\s+BAŞKANI\s+',
    r'^.*?\b(?:MECL[İI]S\s+)?GRUP\s+BAŞKAN\s+VEK[İI]L[İI]\s+',
]

def clean_speaker_name(raw):
    if looks_like_bad_label(raw):
        return pd.NA
    text = re.sub(r'\([^()]*\)', ' ', norm_space(raw))
    text = strip_prefix_using_upper(text, GROUP_PREFIX_PATTERN)
    for _ in range(2):
        for pattern in PREFIX_PATTERNS:
            stripped = strip_prefix_using_upper(text, pattern)
            if stripped != text:
                text = stripped
                break
    text = re.sub(r'\bADINA\b', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'[.:;,\-–—]+$', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return tr_upper(text) if text else pd.NA

NO_CITY_TYPES = {
    'başkan', 'başkan vekili', 'tbmm başkanı', 'cumhurbaşkanı',
    'cumhurbaşkanı yardımcısı', 'bakan', 'devlet başkanı',
    'kamu başdenetçisi', 'parse_error'
}

def mode_or_na(series):
    values = series.dropna()
    return values.mode().iloc[0] if not values.empty else pd.NA

In [ ]:
# Ham konuşma tablosunu zenginleştir.
enriched = df.copy()
enriched['konusmaci_ad_soyad'] = enriched['konusmaci_raw'].apply(clean_speaker_name)
enriched['konusmaci_tip'] = enriched['konusmaci_raw'].apply(classify_speaker_type)
enriched['konusmaci_parti'] = enriched['konusmaci_raw'].apply(normalize_party)
enriched['konusmaci_sehir'] = enriched['konusmaci_raw'].apply(extract_city_from_raw)

# (Devamla) satırlarında aynı ad-soyad için tutanakta görülen en sık şehri kullan.
name_to_city = (
    enriched.loc[
        enriched['konusmaci_ad_soyad'].notna() & enriched['konusmaci_sehir'].notna(),
        ['konusmaci_ad_soyad', 'konusmaci_sehir']
    ]
    .groupby('konusmaci_ad_soyad')['konusmaci_sehir']
    .agg(mode_or_na)
)
city_fill_mask = (
    enriched['konusmaci_sehir'].isna()
    & enriched['konusmaci_ad_soyad'].notna()
    & ~enriched['konusmaci_tip'].isin(NO_CITY_TYPES)
)
enriched.loc[city_fill_mask, 'konusmaci_sehir'] = (
    enriched.loc[city_fill_mask, 'konusmaci_ad_soyad'].map(name_to_city)
)


def find_column(columns, candidates):
    normalized = {str(c).strip().casefold(): c for c in columns}
    for candidate in candidates:
        if candidate.casefold() in normalized:
            return normalized[candidate.casefold()]
    return None

def read_lookup(path):
    suffix = path.suffix.casefold()
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    if suffix == '.parquet':
        return pd.read_parquet(path)
    return pd.read_csv(path, low_memory=False)

# Mevcut milletvekili lookup dosyasından parti ve şehir bilgisini tamamla.
if LOOKUP_PATH and LOOKUP_PATH.exists():
    lookup_raw = read_lookup(LOOKUP_PATH)
    lookup_name_col = find_column(
        lookup_raw.columns,
        ['ad_soyad', 'ad soyad', 'konusmaci_ad_soyad', 'konusmaci_clean',
         'isim', 'name', 'milletvekili', 'milletvekili_ad_soyad']
    )
    lookup_party_col = find_column(
        lookup_raw.columns, ['parti', 'party', 'siyasi_parti', 'siyasi parti']
    )
    lookup_city_col = find_column(
        lookup_raw.columns, ['sehir', 'şehir', 'il', 'city']
    )

    if lookup_name_col is None:
        raise ValueError('Lookup dosyasında ad-soyad sütunu bulunamadı.')

    lookup = pd.DataFrame({'_name_key': lookup_raw[lookup_name_col].apply(clean_key)})
    if lookup_party_col is not None:
        normalized_party = lookup_raw[lookup_party_col].apply(normalize_party)
        original_party = lookup_raw[lookup_party_col].apply(norm_space).replace('', pd.NA)
        lookup['_lookup_party'] = normalized_party.fillna(original_party)
    if lookup_city_col is not None:
        lookup['_lookup_city'] = lookup_raw[lookup_city_col].apply(canonical_city)

    lookup = lookup[lookup['_name_key'].ne('')].drop_duplicates('_name_key', keep='first')
    enriched['_name_key'] = enriched['konusmaci_ad_soyad'].apply(clean_key)
    enriched = enriched.merge(lookup, on='_name_key', how='left')

    if '_lookup_party' in enriched.columns:
        party_fill_mask = (
            enriched['konusmaci_parti'].isna()
            & enriched['_lookup_party'].notna()
            & enriched['konusmaci_ad_soyad'].notna()
        )
        enriched.loc[party_fill_mask, 'konusmaci_parti'] = enriched.loc[party_fill_mask, '_lookup_party']

    if '_lookup_city' in enriched.columns:
        city_fill_mask = (
            enriched['konusmaci_sehir'].isna()
            & enriched['_lookup_city'].notna()
            & ~enriched['konusmaci_tip'].isin(NO_CITY_TYPES)
        )
        enriched.loc[city_fill_mask, 'konusmaci_sehir'] = enriched.loc[city_fill_mask, '_lookup_city']

    enriched = enriched.drop(
        columns=['_name_key', '_lookup_party', '_lookup_city'], errors='ignore'
    )
    print(f'Lookup kullanıldı: {LOOKUP_PATH} ({len(lookup):,} benzersiz kişi)')
else:
    print(f'Uyarı: lookup bulunamadı, lookup eşleştirmesi atlandı: {LOOKUP_PATH}')

# Kurumsal görevlerde seçim çevresi/şehir kullanma.
enriched.loc[enriched['konusmaci_tip'].isin(NO_CITY_TYPES), 'konusmaci_sehir'] = pd.NA
enriched.loc[enriched['konusmaci_tip'].eq('parse_error'), ['konusmaci_ad_soyad', 'konusmaci_parti']] = pd.NA

Lookup kullanıldı: /content/drive/MyDrive/tbmm_raw_tutanaklar/milletvekili_party_city_lookup.csv (599 benzersiz kişi)


In [ ]:
# İstenen kesin sütun sırası: date sonrasında kaynakta bulunan bütün URL sütunları.
final_columns = [
    'session_id', 'donem', 'yasama_yili', 'birlesim_no', 'date',
    *url_columns,
    'konusmaci_raw', 'konusmaci_ad_soyad', 'konusma_text',
    'konusma_sirasi', 'konusma_uzunlugu', 'kelime_sayisi',
    'konusmaci_parti', 'konusmaci_sehir', 'konusmaci_tip'
]

# Aynı isimli bir sütun yanlışlıkla iki kez seçilmesin.
final_columns = list(dict.fromkeys(final_columns))
missing_final_columns = [c for c in final_columns if c not in enriched.columns]
if missing_final_columns:
    raise ValueError(f'Final tablo oluşturulamadı; eksik sütunlar: {missing_final_columns}')

final_table = enriched.loc[:, final_columns].copy()

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
final_table.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'Final tablo kaydedildi: {OUTPUT_CSV}')
print(f'Boyut: {final_table.shape[0]:,} satır x {final_table.shape[1]} sütun')
display(final_table.head(10))

Final tablo kaydedildi: /content/drive/MyDrive/tbmm_raw_tutanaklar/tbmm_konusma_final_table.csv
Boyut: 199,158 satır x 18 sütun


,session_id,donem,yasama_yili,birlesim_no,date,list_page_url,ham_page_url,ozet_page_url,ham_html_url,konusmaci_raw,konusmaci_ad_soyad,konusma_text,konusma_sirasi,konusma_uzunlugu,kelime_sayisi,konusmaci_parti,konusmaci_sehir,konusmaci_tip
0,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,Türkiye Büyük Millet Meclisinin 28’inci Yasama...,1,138,17,MHP,<NA>,başkan
1,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,"Sayın milletvekilleri, Sayın Cumhurbaşkanı ken...",2,314,35,MHP,<NA>,başkan
2,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,"Sayın milletvekilleri, şimdi gündeme geçiyoruz...",3,196,24,MHP,<NA>,başkan
3,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,"Muhterem Cumhurbaşkanım, değerli milletvekille...",4,3833,477,MHP,<NA>,başkan
4,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,Esas gündemimiz olan ant içme törenine geçiyor...,5,1684,204,MHP,<NA>,başkan
5,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,İstanbul Milletvekili Sayın Yücel Arzen Hacıoğ...,6,144,17,MHP,<NA>,başkan
6,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,Şimdi ant içmek üzere sırasıyla milletvekiller...,7,667,92,MHP,<NA>,başkan
7,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,"Sayın milletvekilleri, ant içme sırası Ankara ...",8,268,33,MHP,<NA>,başkan
8,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,"Sağ olun efendim. Sayın milletvekilleri, Sayın...",9,193,23,MHP,<NA>,başkan
9,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,"Sayın milletvekilleri, ant içmeye Ankara ili m...",10,390,55,MHP,<NA>,başkan


In [ ]:
# Son kontroller
assert len(final_table) == len(df), 'Satır sayısı değişti.'
assert final_table.columns.tolist() == final_columns, 'Sütun sırası beklenenden farklı.'
assert final_table.loc[
    final_table['konusmaci_tip'].isin(NO_CITY_TYPES), 'konusmaci_sehir'
].isna().all(), 'Kurumsal görevlerden bazılarında şehir dolu kaldı.'

print('Final sütunlar:')
print(final_table.columns.tolist())

print('\nEksik değer sayıları:')
display(final_table.isna().sum().to_frame('eksik_sayi'))

print('\nKonuşmacı türleri:')
display(final_table['konusmaci_tip'].value_counts(dropna=False).to_frame('sayi'))

Final sütunlar:
['session_id', 'donem', 'yasama_yili', 'birlesim_no', 'date', 'list_page_url', 'ham_page_url', 'ozet_page_url', 'ham_html_url', 'konusmaci_raw', 'konusmaci_ad_soyad', 'konusma_text', 'konusma_sirasi', 'konusma_uzunlugu', 'kelime_sayisi', 'konusmaci_parti', 'konusmaci_sehir', 'konusmaci_tip']

Eksik değer sayıları:


,eksik_sayi
session_id,0
donem,0
yasama_yili,0
birlesim_no,0
date,0
list_page_url,0
ham_page_url,0
ozet_page_url,0
ham_html_url,0
konusmaci_raw,0



Konuşmacı türleri:


,sayi
konusmaci_tip,
milletvekili,115919
başkan,70740
milletvekili (grup adına),7506
bakan,2552
komisyon sözcüsü,1091
cumhurbaşkanı yardımcısı,511
komisyon başkanı,473
parse_error,200
komisyon başkan vekili,63


In [ ]:
error_rows = final_table[
    (final_table['konusmaci_parti'].isna() | final_table['konusmaci_sehir'].isna()) &
    (final_table['konusmaci_tip'] != 'başkan')
]
display(error_rows)

,session_id,donem,yasama_yili,birlesim_no,date,list_page_url,ham_page_url,ozet_page_url,ham_html_url,konusmaci_raw,konusmaci_ad_soyad,konusma_text,konusma_sirasi,konusma_uzunlugu,kelime_sayisi,konusmaci_parti,konusmaci_sehir,konusmaci_tip
65,28_1_2,28,1,2,2023-06-03,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,CUMHURBAŞKANI RECEP TAYYİP ERDOĞAN,RECEP TAYYİP ERDOĞAN,"Hoş bulduk, sağ olun. (Cumhurbaşkanı Recep Tay...",3,331,40,<NA>,<NA>,cumhurbaşkanı
67,28_1_2,28,1,2,2023-06-03,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,CUMHURBAŞKANI RECEP TAYYİP ERDOĞAN,RECEP TAYYİP ERDOĞAN,"""Cumhurbaşkanı sıfatıyla, Devletin varlığı ve ...",5,775,93,<NA>,<NA>,cumhurbaşkanı
1054,28_1_8,28,1,8,2023-06-21,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,SÜMEYYE BOZ (Muş),SÜMEYYE BOZ,"“Güvenlik” adı altında Şırnak, Amed ve son ola...",23,758,100,<NA>,Muş,milletvekili
1082,28_1_8,28,1,8,2023-06-21,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,ÇİĞDEM ERDOĞAN ATABEK (Sakarya),ÇİĞDEM ERDOĞAN ATABEK,"Sayın Başkan, değerli milletvekilleri; bugün b...",51,851,106,<NA>,Sakarya,milletvekili
1301,28_1_9,28,1,9,2023-06-22,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,SÜMEYYE BOZ (Muş),SÜMEYYE BOZ,Birkaç hafta önce Muş’ta 12 arkadaşımız 23 Ara...,51,677,87,<NA>,Muş,milletvekili
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198985,28_4_96,28,4,96,2026-05-20,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/4/9...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/4/9...,KEZBAN KONUKÇU (İstanbul),KEZBAN KONUKÇU,"Sevgili arkadaşlar, hepimizin bildiği gibi on ...",531,3816,478,<NA>,İstanbul,milletvekili
198986,28_4_96,28,4,96,2026-05-20,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/4/9...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/4/9...,KEZBAN KONUKÇU (Devamla),KEZBAN KONUKÇU,Tamamlıyorum.,532,13,1,<NA>,İstanbul,milletvekili
198988,28_4_96,28,4,96,2026-05-20,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/4/9...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/4/9...,KEZBAN KONUKÇU (Devamla),KEZBAN KONUKÇU,"Ayrıca, üretim faaliyeti gerekçesiyle yapılan ...",534,902,112,<NA>,İstanbul,milletvekili
199102,28_4_96,28,4,96,2026-05-20,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/4/9...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/4/9...,SÜMEYYE BOZ (Muş),SÜMEYYE BOZ,"Teşekkürler Başkan. Değerli milletvekilleri, h...",648,3949,513,<NA>,Muş,milletvekili


In [ ]:
exclude_condition = (
    ((final_table['konusmaci_tip'] == 'milletvekili') & (final_table['konusmaci_parti'].isna())) |
    (final_table['konusmaci_tip'] == 'parse_error')
)

final_table_without_missing_milletvekili_parti = final_table[~exclude_condition]

output_filename = '/content/drive/MyDrive/tbmm_raw_tutanaklar/donem28_konusmaci_bazli.csv'
final_table_without_missing_milletvekili_parti.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f'DataFrame saved to: {output_filename}')
display(final_table_without_missing_milletvekili_parti.head())

DataFrame saved to: /content/drive/MyDrive/tbmm_raw_tutanaklar/donem28_konusmaci_bazli.csv


,session_id,donem,yasama_yili,birlesim_no,date,list_page_url,ham_page_url,ozet_page_url,ham_html_url,konusmaci_raw,konusmaci_ad_soyad,konusma_text,konusma_sirasi,konusma_uzunlugu,kelime_sayisi,konusmaci_parti,konusmaci_sehir,konusmaci_tip
0,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,Türkiye Büyük Millet Meclisinin 28’inci Yasama...,1,138,17,MHP,<NA>,başkan
1,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,"Sayın milletvekilleri, Sayın Cumhurbaşkanı ken...",2,314,35,MHP,<NA>,başkan
2,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,"Sayın milletvekilleri, şimdi gündeme geçiyoruz...",3,196,24,MHP,<NA>,başkan
3,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,"Muhterem Cumhurbaşkanım, değerli milletvekille...",4,3833,477,MHP,<NA>,başkan
4,28_1_1,28,1,1,2023-06-02,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,Esas gündemimiz olan ant içme törenine geçiyor...,5,1684,204,MHP,<NA>,başkan
